In [1]:
import pandas as pd
import chardet

with open("../data/raw/ecos_ppi_egg_feed_electricity_2016_2025.csv", "rb") as f:
    result_ecos = chardet.detect(f.read())

ecos_df = pd.read_csv("../data/raw/ecos_ppi_egg_feed_electricity_2016_2025.csv",
                       encoding=result_ecos['encoding'])

print(ecos_df.shape)
print(ecos_df.columns.tolist())
print(ecos_df.head())

(4, 125)
['통계표', '항목명1', '단위', '가중치', '변환', '2016/01', '2016/02', '2016/03', '2016/04', '2016/05', '2016/06', '2016/07', '2016/08', '2016/09', '2016/10', '2016/11', '2016/12', '2017/01', '2017/02', '2017/03', '2017/04', '2017/05', '2017/06', '2017/07', '2017/08', '2017/09', '2017/10', '2017/11', '2017/12', '2018/01', '2018/02', '2018/03', '2018/04', '2018/05', '2018/06', '2018/07', '2018/08', '2018/09', '2018/10', '2018/11', '2018/12', '2019/01', '2019/02', '2019/03', '2019/04', '2019/05', '2019/06', '2019/07', '2019/08', '2019/09', '2019/10', '2019/11', '2019/12', '2020/01', '2020/02', '2020/03', '2020/04', '2020/05', '2020/06', '2020/07', '2020/08', '2020/09', '2020/10', '2020/11', '2020/12', '2021/01', '2021/02', '2021/03', '2021/04', '2021/05', '2021/06', '2021/07', '2021/08', '2021/09', '2021/10', '2021/11', '2021/12', '2022/01', '2022/02', '2022/03', '2022/04', '2022/05', '2022/06', '2022/07', '2022/08', '2022/09', '2022/10', '2022/11', '2022/12', '2023/01', '2023/02', '2023/03',

In [2]:
print(ecos_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Columns: 125 entries, 통계표 to 2025/12
dtypes: float64(121), str(4)
memory usage: 4.0 KB
None


In [3]:
print(ecos_df.isnull().sum())

통계표        0
항목명1       0
단위         0
가중치        0
변환         0
          ..
2025/08    0
2025/09    0
2025/10    0
2025/11    0
2025/12    0
Length: 125, dtype: int64


In [4]:
print(ecos_df[['통계표']].values)

[['4.1.1.3. 생산자물가지수(품목별)']
 ['4.2.1. 소비자물가지수']
 ['4.1.1.3. 생산자물가지수(품목별)']
 ['4.1.1.3. 생산자물가지수(품목별)']]


In [5]:
print(ecos_df['항목명1'].values)

<StringArray>
['      달걀', '        달걀', '      양계용배합사료', '      산업용전력']
Length: 4, dtype: str


In [6]:
ecos_df['지표명'] = ecos_df['통계표'].str.strip() + '_' + ecos_df['항목명1'].str.strip()
print(ecos_df['지표명'].values)

<StringArray>
[     '4.1.1.3. 생산자물가지수(품목별)_달걀',             '4.2.1. 소비자물가지수_달걀',
 '4.1.1.3. 생산자물가지수(품목별)_양계용배합사료',   '4.1.1.3. 생산자물가지수(품목별)_산업용전력']
Length: 4, dtype: str


/var/folders/8j/ww9j2kgd3rz_3f49qz8hm3dh0000gn/T/ipykernel_45669/984235559.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ecos_df['지표명'] = ecos_df['통계표'].str.strip() + '_' + ecos_df['항목명1'].str.strip()


In [7]:
date_cols = [col for col in ecos_df.columns if '/' in str(col)]

ecos_long_df = ecos_df[['지표명'] + date_cols].melt(
    id_vars='지표명',
    var_name='year_month',
    value_name='value'
)

ecos_long_df['year_month'] = pd.to_datetime(ecos_long_df['year_month'], format='%Y/%m')

ecos_final_df = ecos_long_df.pivot(
    index='year_month',
    columns='지표명',
    values='value'
).reset_index()

ecos_final_df.columns.name = None
ecos_final_df = ecos_final_df.rename(columns={
    '4.1.1.3. 생산자물가지수(품목별)_달걀'    : 'egg_ppi',
    '4.2.1. 소비자물가지수_달걀'              : 'egg_cpi',
    '4.1.1.3. 생산자물가지수(품목별)_양계용배합사료' : 'feed_ppi',
    '4.1.1.3. 생산자물가지수(품목별)_산업용전력'    : 'electricity_ppi'
})

print(ecos_final_df.shape)   
print(ecos_final_df.head())
print(ecos_final_df.isnull().sum())

(120, 5)
  year_month  egg_ppi  electricity_ppi  feed_ppi  egg_cpi
0 2016-01-01    92.17            98.81     95.69   92.594
1 2016-02-01    85.08            98.81     95.69   91.098
2 2016-03-01    83.52            98.81     94.82   87.269
3 2016-04-01    89.81            98.81     94.82   86.773
4 2016-05-01    85.53            98.81     94.82   84.645
year_month         0
egg_ppi            0
electricity_ppi    0
feed_ppi           0
egg_cpi            0
dtype: int64


In [8]:
kape_egg_df = pd.read_html("../data/raw/kape_egg_retail_price_2016_2025.xls",
                  encoding='utf-8')[0]

print(kape_egg_df.shape)

(122, 21)


In [9]:
print(kape_egg_df.columns.tolist())
print(kape_egg_df.head(2))
print(kape_egg_df.tail(2))  

[('소비자 기간별 가격()', '구분'), ('소비자 기간별 가격()', '평균'), ('소비자 기간별 가격()', '최고'), ('소비자 기간별 가격()', '최저'), ('소비자 기간별 가격()', '서울'), ('소비자 기간별 가격()', '부산'), ('소비자 기간별 가격()', '대구'), ('소비자 기간별 가격()', '인천'), ('소비자 기간별 가격()', '광주'), ('소비자 기간별 가격()', '대전'), ('소비자 기간별 가격()', '울산'), ('소비자 기간별 가격()', '세종'), ('소비자 기간별 가격()', '경기'), ('소비자 기간별 가격()', '강원'), ('소비자 기간별 가격()', '충북'), ('소비자 기간별 가격()', '충남'), ('소비자 기간별 가격()', '전북'), ('소비자 기간별 가격()', '전남'), ('소비자 기간별 가격()', '경북'), ('소비자 기간별 가격()', '경남'), ('소비자 기간별 가격()', '제주')]
  소비자 기간별 가격()                                                        ...  \
            구분    평균    최고    최저    서울    부산    대구    인천    광주    대전  ...   
0      25년 12월  6880  7720  6106  6667  7178  6607  6737  6442  7455  ...   
1      25년 11월  6499  7387  5634  6679  6347  5916  7000  6795  6548  ...   

                                                               
     세종    경기    강원    충북    충남    전북    전남    경북    경남    제주  
0  7702  6721  7382  6106  7182  6536  6816  6563  6701  7

In [10]:
kape_egg_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   (소비자 기간별 가격(), 구분)  122 non-null    str  
 1   (소비자 기간별 가격(), 평균)  122 non-null    int64
 2   (소비자 기간별 가격(), 최고)  122 non-null    int64
 3   (소비자 기간별 가격(), 최저)  122 non-null    int64
 4   (소비자 기간별 가격(), 서울)  122 non-null    str  
 5   (소비자 기간별 가격(), 부산)  122 non-null    str  
 6   (소비자 기간별 가격(), 대구)  122 non-null    str  
 7   (소비자 기간별 가격(), 인천)  122 non-null    str  
 8   (소비자 기간별 가격(), 광주)  122 non-null    str  
 9   (소비자 기간별 가격(), 대전)  122 non-null    str  
 10  (소비자 기간별 가격(), 울산)  122 non-null    str  
 11  (소비자 기간별 가격(), 세종)  122 non-null    str  
 12  (소비자 기간별 가격(), 경기)  122 non-null    str  
 13  (소비자 기간별 가격(), 강원)  122 non-null    str  
 14  (소비자 기간별 가격(), 충북)  122 non-null    str  
 15  (소비자 기간별 가격(), 충남)  122 non-null    str  
 16  (소비자 기간별 가격(), 전북)  122 non-null    str  
 17  (소비자 기간별

In [11]:
kape_egg_df_clean = kape_egg_df[[('소비자 기간별 가격()', '구분'), 
               ('소비자 기간별 가격()', '평균')]].copy()

kape_egg_df_clean.columns = ['year_month', 'egg_price']

kape_egg_df_clean = kape_egg_df_clean[~kape_egg_df_clean['year_month'].isin(['전년', '평년'])]

kape_egg_df_clean['egg_price'] = pd.to_numeric(kape_egg_df_clean['egg_price'], errors='coerce')

print(kape_egg_df_clean.shape)   
print(kape_egg_df_clean.head())
print(kape_egg_df_clean.tail())
print(kape_egg_df_clean.isnull().sum())  

(120, 2)
  year_month  egg_price
0    25년 12월       6880
1    25년 11월       6499
2    25년 10월       7039
3    25년 09월       6687
4    25년 08월       7088
    year_month  egg_price
115    16년 05월       5216
116    16년 04월       5259
117    16년 03월       5260
118    16년 02월       5473
119    16년 01월       5493
year_month    0
egg_price     0
dtype: int64


In [12]:
kape_egg_df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   year_month  120 non-null    str  
 1   egg_price   120 non-null    int64
dtypes: int64(1), str(1)
memory usage: 2.0 KB


In [13]:
kape_egg_df_clean['year_month'] = kape_egg_df_clean['year_month'].str.replace('년 ', '-').str.replace('월', '').str.strip()

kape_egg_df_clean['year_month'] = kape_egg_df_clean['year_month'].apply(
    lambda x: '20' + x if int(x.split('-')[0]) < 100 else x
)

kape_egg_df_clean['year_month'] = pd.to_datetime(kape_egg_df_clean['year_month'], format='%Y-%m')

kape_egg_df_clean = kape_egg_df_clean.sort_values('year_month').reset_index(drop=True)

print(kape_egg_df_clean.dtypes)

year_month    datetime64[us]
egg_price              int64
dtype: object


In [14]:
# KAPE 중복 확인
print("=== KAPE ===")
print("year_month 중복 수:", kape_egg_df_clean['year_month'].duplicated().sum())
print("고유값 수:", kape_egg_df_clean['year_month'].nunique())
print("전체 행 수:", len(kape_egg_df_clean))

# ECOS 중복 확인
print("\n=== ECOS ===")
print("year_month 중복 수:", ecos_final_df['year_month'].duplicated().sum())
print("고유값 수:", ecos_final_df['year_month'].nunique())
print("전체 행 수:", len(ecos_final_df))

=== KAPE ===
year_month 중복 수: 0
고유값 수: 120
전체 행 수: 120

=== ECOS ===
year_month 중복 수: 0
고유값 수: 120
전체 행 수: 120


In [15]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from src.db.connection import engine

kape_egg_df_clean.to_sql( "kamis_egg_retail", engine, if_exists="replace", index=False)

ecos_final_df.to_sql( "ecos_ppi", engine, if_exists="replace", index=False)

print("적재 완료")

# 검증 
df_kamis_db = pd.read_sql("SELECT * FROM kamis_egg_retail", engine)
df_ecos_db  = pd.read_sql("SELECT * FROM ecos_ppi", engine)

print("KAPE CSV 행 수:", len(kape_egg_df_clean))
print("KAPE DB  행 수:", len(df_kamis_db))
print("일치 여부:", len(kape_egg_df_clean) == len(df_kamis_db))

print("\nECOS CSV 행 수:", len(ecos_final_df))
print("ECOS DB  행 수:", len(df_ecos_db))
print("일치 여부:", len(ecos_final_df) == len(df_ecos_db))

적재 완료
KAPE CSV 행 수: 120
KAPE DB  행 수: 120
일치 여부: True

ECOS CSV 행 수: 120
ECOS DB  행 수: 120
일치 여부: True
